In [1]:
import pandas as pd
import xarray as xr

In [2]:
ds = xr.open_dataset("/Volumes/T7/Data/EOBS/fg_ens_mean_0.1deg_reg_v31.0e.nc")
ds

<xarray.Dataset> Size: 22GB
Dimensions:    (time: 16437, longitude: 705, latitude: 465)
Coordinates:
  * time       (time) datetime64[ns] 131kB 1980-01-01 1980-01-02 ... 2024-12-31
  * longitude  (longitude) float64 6kB -24.95 -24.85 -24.75 ... 45.35 45.45
  * latitude   (latitude) float64 4kB 25.05 25.15 25.25 ... 71.25 71.35 71.45
Data variables:
    fg         (time, latitude, longitude) float32 22GB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.4 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Tue Mar 11 10:44:57 2025: cdo -z zip_3 mergetime /nobackup_...
    CDO:          Climate Data Operators version 2.4.4 (https://mpimet.mpg.de...

In [3]:
ds_sel = ds.sel(time=slice("1960-01-01", None))

In [4]:
def extract_eobs_timeseries(ds, var_name, lat, lon, start=None, end=None):
    """
    Extract a time series from an E-OBS dataset at a given location
    and within a selected time period.

    Parameters
    ----------
    ds : xarray.Dataset
        The loaded E-OBS dataset.
    var_name : str
        Name of the variable to extract, e.g. "tg".
    lat : float
        Latitude of the desired location.
    lon : float
        Longitude of the desired location.
    start : str or None
        Start date in format 'YYYY-MM-DD' (e.g. '1960-01-01').
        If None, no lower time bound is applied.
    end : str or None
        End date in format 'YYYY-MM-DD'.
        If None, no upper time bound is applied.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing time, lat, lon, and the variable values.
    """

    # Select time range if provided
    if start or end:
        ds = ds.sel(time=slice(start, end))

    # Extract nearest grid point for the variable
    ts = ds[var_name].sel(latitude=lat, longitude=lon, method="nearest")

    # Convert to DataFrame
    df = ts.to_dataframe().reset_index()

    return df

In [5]:
def get_minimum_necessary_df(large_df, variable):
    df = pd.DataFrame()

    df["time"] = large_df["time"]
    df[variable] = large_df[variable]

    return df

In [6]:
locations = {
    "Cluj": (46.75, 23.65),
    "Gheorgheni": (46.75, 25.45),
    "Brasov": (45.75, 25.55),
    "Deva": (45.85, 22.95),
    "Pecs": (46.05, 18.25),
    "Gyor": (47.75, 17.65),
    "Oradea": (47.15, 21.85),
    "Kassa": (48.65, 21.25),
    "Kecskemet": (46.85, 19.75),
    "Keszthely": (46.75, 17.15),
    "Bacskatopolya": (45.85, 19.65),
}

In [7]:
current_variable = "fg"

for key in locations:

    print(f"Working with {key} : {locations[key]}...")

    df_tn_loc = extract_eobs_timeseries(
        ds,
        var_name=current_variable,
        lat=locations[key][0],
        lon=locations[key][1],
        start="1960-01-01",
        end=None
    )

    print(f"Got large df")

    df_min_loc = get_minimum_necessary_df(df_tn_loc, current_variable)

    print(f"Small df is ready")

    df_min_loc.to_csv(f"../{current_variable}/{key}_{current_variable}.csv", index = False)

    print(f"Data saved to ../{current_variable}/{key}_{current_variable}.csv")

    print(f"\n\n\n")

Working with Cluj : (46.75, 23.65)...
Got large df
Small df is ready
Data saved to ../fg/Cluj_fg.csv




Working with Gheorgheni : (46.75, 25.45)...
Got large df
Small df is ready
Data saved to ../fg/Gheorgheni_fg.csv




Working with Brasov : (45.75, 25.55)...
Got large df
Small df is ready
Data saved to ../fg/Brasov_fg.csv




Working with Deva : (45.85, 22.95)...
Got large df
Small df is ready
Data saved to ../fg/Deva_fg.csv




Working with Pecs : (46.05, 18.25)...
Got large df
Small df is ready
Data saved to ../fg/Pecs_fg.csv




Working with Gyor : (47.75, 17.65)...
Got large df
Small df is ready
Data saved to ../fg/Gyor_fg.csv




Working with Oradea : (47.15, 21.85)...
Got large df
Small df is ready
Data saved to ../fg/Oradea_fg.csv




Working with Kassa : (48.65, 21.25)...
Got large df
Small df is ready
Data saved to ../fg/Kassa_fg.csv




Working with Kecskemet : (46.85, 19.75)...
Got large df
Small df is ready
Data saved to ../fg/Kecskemet_fg.csv




Working with Keszthely 